# pandas 之六（补）：arrow 时间库

---

## 为什么还要学一个 arrow？

pandas 处理**批量时间序列**很强，但处理**单个时间点**的日常操作有点啰嗦：

```python
# 想知道"三周后是哪天"，pandas 里要写：
pd.Timestamp.now() + pd.Timedelta(weeks=3)

# arrow 里直接：
arrow.now().shift(weeks=3)
```

**定位区别**：

| | 擅长 | 不擅长 |
|---|---|---|
| **pandas** | 一整列时间数据的运算、重采样、分组 | 单个时间的加减、格式化 |
| **arrow** | **单个时间点**的运算、转换、人性化格式 | 批量列运算 |

**一句话**：**批量用 pandas，单个用 arrow。**
两者是互补的，不是替代关系。

> **注意别搞混**：这个 `arrow` 是**时间库**，
> 和另一个同名的 `pyarrow`（大数据列式存储库）**完全是两回事**。

## 学习目标

1. 会创建 / 解析时间点
2. 会做时间的加减（`shift`）
3. 会生成时间序列（`Arrow.range`）
4. 掌握 pandas 里最常用的 4 种"时间偏移"


In [ ]:
import arrow
import pandas as pd

print('arrow 版本：', arrow.__version__)
print('pandas 版本：', pd.__version__)


---
## 一、创建时间点

arrow 里最核心的对象是 `Arrow`。


In [ ]:
# 1) 当前时间（本地时区）
now = arrow.now()
print('arrow.now()      :', now)
print('类型             :', type(now))
print('它的时区         :', now.tzinfo)

# 2) UTC 当前时间
print('\narrow.utcnow()   :', arrow.utcnow())

# 3) 从字符串解析
print("\narrow.get('2023-08-15 10:00') :", arrow.get('2023-08-15 10:00'))
print("arrow.get('2023/08/15')       :", arrow.get('2023/08/15'))

# 4) 直接实例化（年, 月, 日, 时, 分, 秒）
print('\narrow.Arrow(2023, 8, 15, 10, 0, 0) :')
print('  ', arrow.Arrow(2023, 8, 15, 10, 0, 0))


---
## 二、从 Arrow 对象里"取"信息


In [ ]:
t = arrow.get('2023-08-15 10:30:45')
print('时间点：', t)

print('\n=== 直接取属性 ===')
print('年 :', t.year)
print('月 :', t.month)
print('日 :', t.day)
print('时 :', t.hour)
print('分 :', t.minute)
print('秒 :', t.second)

print('\n=== 取"整块" ===')
print('纯日期 :', t.date(), '  类型', type(t.date()).__name__)
print('纯时间 :', t.time(), '  类型', type(t.time()).__name__)


> **`.date()` 和 `.time()` 返回的是 Python 原生 `datetime.date` / `datetime.time`**，
> 这就方便了 —— 可以直接喂给那些"只认标准库"的第三方库。


In [ ]:
print('\n=== 格式化输出（最实用）===')
print("ISO       :", t.isoformat())
print("自定义格式 :", t.format('YYYY年MM月DD日 HH:mm'))
print("只要日期   :", t.format('YYYY-MM-DD'))
print("中文星期   :", t.format('dddd', locale='zh-cn'))


**`format()` 的占位符**（和 Python 的 `strftime` 很像但更可读）：

| 占位符 | 含义 | 例子 |
|---|---|---|
| `YYYY` | 4 位年 | 2023 |
| `MM` | 月 | 08 |
| `DD` | 日 | 15 |
| `HH` | 24 小时制 | 10 |
| `mm` | 分 | 30 |
| `dddd` | 星期全名 | Tuesday |

> **注意大小写**：`MM` 是月，`mm` 是分！写反了结果会很离谱。


---
## 三、时间的加减：`shift`

**`shift` 是 arrow 最好用的功能**——加时间像说人话一样自然。


In [ ]:
t = arrow.get('2023-08-15 10:00')
print('基准时间：', t)

print('\n=== 往后 ===')
print('3 周后     :', t.shift(weeks=3))
print('2 天后     :', t.shift(days=2))
print('5 小时后   :', t.shift(hours=5))
print('90 分钟后  :', t.shift(minutes=90))

print('\n=== 往前（负数）===')
print('1 个月前   :', t.shift(months=-1))
print('10 天前    :', t.shift(days=-10))

print('\n=== 组合（可以一次传多个）===')
print('明年上月的今天 :', t.shift(years=1, months=-1))
print('3周零2天后     :', t.shift(weeks=3, days=2))


> **`shift` 支持的单位**：`years` / `months` / `weeks` / `days` /
> `hours` / `minutes` / `seconds` / `microseconds`，
> 都可以一次传多个，正数往后、负数往前。


### 对比：同一件事，arrow 和 pandas 怎么写


In [ ]:
# arrow 写法
print('arrow :', arrow.get('2023-08-15').shift(months=-1))

# pandas 写法
print('pandas:', pd.Timestamp('2023-08-15') + pd.DateOffset(months=-1))


都行。但 arrow 的写法**一眼就懂**，而 pandas 需要你记住 `DateOffset` 这个名字。


### `humanize()`：把时间差变成"人话"


In [ ]:
import datetime

base = arrow.get('2023-08-15 10:00')
print('相对于基准时间：')
for delta in [{'hours': -3}, {'days': -1}, {'days': -7}, {'months': -2}]:
    other = base.shift(**delta)
    print(f'  {str(delta):22} -> {other.humanize(base, locale="zh-cn")}')


**`humanize` 会输出"3小时前""2个月前"这种表达**，
做"最后登录时间"这类界面文案时特别省事。


---
## 四、生成时间序列：`Arrow.range`

arrow 也能生成序列，但**返回的是生成器**，要看内容必须 `list()` 转换。

> **原稿的提示**：原稿打印 `list(a)` 是对的，但没说明
> "不 `list()` 就只能看到一个 generator 对象"这个坑。
> 这里把它讲清楚。


In [ ]:
# 从今天起，连续 8 天
a = arrow.Arrow.range('day', arrow.now(), arrow.now().shift(days=7))
print('不加 list() 看到的是：', a, '  <- 生成器，直接打印看不到内容')
print('\n加 list() 才能看到列表：')
print([x.format('YYYY-MM-DD') for x in a])


In [ ]:
# 指定起止日期
b = arrow.Arrow.range('day', arrow.get('2021-01-01'), arrow.get('2021-01-10'))
print('2021-01-01 到 2021-01-10 每天：')
print([x.format('YYYY-MM-DD') for x in b])


### `span` / `floor` / `ceil`：取某个时间区间的边界


In [ ]:
t = arrow.get('2023-08-15 14:30:45')
print('基准时间：', t)
print('所在月份区间 :', t.span('month'))
print('本月第一天   :', t.floor('month'))
print('本月最后一刻 :', t.ceil('month'))
print('所在周区间   :', t.span('week'))


> **`floor` / `ceil` 在报表里非常有用**：
> "本月第一天""本月最后一天"这类需求，一行就出来了。
> 对比 pandas 要写 `pd.offsets.MonthEnd` 那一串，arrow 更直观。


---
## 五、pandas 的时间偏移全家桶（4 种）

前面在时间序列那篇提过，这里系统整理一遍。

| 写法 | 类型 | 行为 | 适合 |
|---|---|---|---|
| `+ pd.Timedelta(days=7)` | 时长 | 就是 7×24 小时 | 天/小时级固定时长 |
| `+ pd.DateOffset(days=7)` | 偏移 | 同上，但支持"月/年" | 需要按日历算 |
| `+ pd.offsets.MonthEnd(1)` | 月末 | 跳到**下一个月末** | 财务月结 |
| `+ pd.offsets.QuarterEnd(1)` | 季末 | 跳到**下一个季末** | 季报 |


In [ ]:
df = pd.DataFrame({'date': pd.date_range('2023-01-15', periods=4, freq='ME')})
print('基准日期：')
print(df)

df['+7天'] = df['date'] + pd.Timedelta(days=7)
df['+7天offset'] = df['date'] + pd.DateOffset(days=7)
df['下月末'] = df['date'] + pd.offsets.MonthEnd(n=1)
df['下季末'] = df['date'] + pd.offsets.QuarterEnd(n=1)

print('\n各种偏移的结果：')
print(df)


### ⚠️ 关键区别：`Timedelta` 和 `MonthEnd` 完全不是一回事

看"下月末"那列：`1月31日 -> 2月28日`。
但如果用 `Timedelta(days=28)`，那就是机械地加 28 天，
**1 月 31 日加 28 天会得到 2 月 28 日**（碰巧对），
而 `3 月 31 日加 28 天`会得到 `4 月 28 日`——**不是月末**，就错了。

**凡是"月末/季末/年末"的需求，必须用 `offsets`。**


In [ ]:
demo = pd.DataFrame({'date': pd.to_datetime(['2023-01-31', '2023-02-28', '2023-04-30'])})
demo['Timedelta(28天)'] = demo['date'] + pd.Timedelta(days=28)
demo['MonthEnd(1)'] = demo['date'] + pd.offsets.MonthEnd(n=1)
print(demo)
print('\n  <- 看 Timedelta 那列：2月28日+28天得到 3月28日，根本不是月末！')


### 常用 offsets 速查

| 写法 | 含义 |
|---|---|
| `pd.offsets.Day(n)` | n 天后 |
| `pd.offsets.MonthEnd(n)` | 第 n 个月末 |
| `pd.offsets.MonthBegin(n)` | 第 n 个月初 |
| `pd.offsets.QuarterEnd(n)` | 第 n 个季末 |
| `pd.offsets.YearEnd(n)` | 第 n 个年末 |
| `pd.offsets.BDay(n)` | n 个**工作日**后（跳过周末） |
| `pd.offsets.CDay(n)` | n 个**自然日**后（含节假日） |

**`n=0` 表示"本"周期**：`MonthEnd(0)` 就是"本月末"。


In [ ]:
t2 = pd.Timestamp('2023-02-10')
print('基准：', t2.date())
print('本周末   MonthEnd(0)  :', (t2 + pd.offsets.MonthEnd(0)).date())
print('本季末   QuarterEnd(0):', (t2 + pd.offsets.QuarterEnd(0)).date())
print('下月末   MonthEnd(1)  :', (t2 + pd.offsets.MonthEnd(1)).date())


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `Arrow.range()` 直接打印 | 显示 `<generator object ...>` | 用 `list()` 或循环取 |
| `format('MM')` / `format('mm')` 写反 | 月变成分钟 | `MM`=月，`mm`=分 |
| 把 `arrow` 和 `pyarrow` 搞混 | 装错库 | `arrow`=时间库，`pyarrow`=列式存储 |
| `shift(month=-1)` 少了 s | `TypeError` | 参数名是 `months`（复数） |
| 用 `Timedelta` 算"月末" | 结果不是月末 | 用 `pd.offsets.MonthEnd` |
| `now.date()` 拿去做时区运算 | 丢了时区信息 | `.date()` 只是纯日期，需要时区就保留 Arrow 对象 |
| 以为 arrow 能替代 pandas | 列运算写不出来 | 批量用 pandas，单个用 arrow |

---

## 七、动手练习

**练习 1**：用 arrow 获取当前时间，打印它的年、月、日、时、分、秒。

**练习 2**：把 `'2023-12-25'` 解析成 Arrow 对象，用 `format` 输出成 `2023年12月25日`。

**练习 3**：计算"从今天起 100 天后"是哪一天，并格式化成 `YYYY-MM-DD`。

**练习 4**：生成 2023 年 1 月 1 日到 1 月 31 日的每一天（用 `Arrow.range`）。

**练习 5**：给定任意一个日期，求它所在月份的第一天和最后一天（提示：`floor` / `ceil`）。

**练习 6**：用 pandas 求 `2023-05-17` 的"本月末""下月末""本季末"，分别用 offsets 实现。

---

**答案思路（先自己做）**

```python
# 1
now = arrow.now()
print(now.year, now.month, now.day, now.hour, now.minute, now.second)

# 2
print(arrow.get('2023-12-25').format('YYYY年MM月DD日'))

# 3
print(arrow.now().shift(days=100).format('YYYY-MM-DD'))

# 4
days = arrow.Arrow.range('day', arrow.get('2023-01-01'), arrow.get('2023-01-31'))
print([d.format('YYYY-MM-DD') for d in days])

# 5
t = arrow.get('2023-05-17')
print(t.floor('month').format('YYYY-MM-DD'), t.ceil('month').format('YYYY-MM-DD'))

# 6
t = pd.Timestamp('2023-05-17')
print((t + pd.offsets.MonthEnd(0)).date())
print((t + pd.offsets.MonthEnd(1)).date())
print((t + pd.offsets.QuarterEnd(0)).date())
```
